In [1]:
import pandas as pd
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
import os
import shutil
import chromadb

In [2]:
cd C:\Users\angel\Desktop\Recommand-System\MOST_committee

C:\Users\angel\Desktop\Recommand-System\MOST_committee


In [3]:
authors_dicts = pd.read_csv('data/research_proj/115計算機學門審查/authors_projects.csv')

In [8]:
# 確保資料庫目錄存在
os.makedirs("database", exist_ok=True)

# 定義向量數據庫路徑 (現在只需要一個路徑)
db_path = "database/vectorstore_bge_merged"

# 刪除現有的向量數據庫（如果存在）
print("正在清理現有的向量數據庫...")
if os.path.exists(db_path):
    print(f"刪除現有的向量數據庫: {db_path}")
    shutil.rmtree(db_path)

# 初始化 BGE 嵌入模型 (明確指定使用 CPU)
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-zh-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# 創建合併後的文檔
title_documents = []
keyword_documents = []

# 遍歷 DataFrame 的每一行
for i, row in authors_dicts.iterrows():
    author = row['manager']
    
    if 'title' in row and not pd.isna(row['title']):
        titles = row['title']
        # 如果標題是字符串形式的列表，需要解析
        if isinstance(titles, str) and titles.startswith('[') and titles.endswith(']'):
            import ast
            try:
                titles = ast.literal_eval(titles)
            except:
                titles = [titles]
        elif not isinstance(titles, list):
            titles = [titles]
            
        all_titles = "\n".join([t for t in titles if t and isinstance(t, str)])
        if all_titles:
            title_doc = Document(
                page_content=all_titles,
                metadata={"manager": author}
            )
            title_documents.append(title_doc)
    
    # 處理關鍵字
    if 'keywords' in row and not pd.isna(row['keywords']):
        keywords = row['keywords']
        # 如果關鍵字是字符串形式的列表，需要解析
        if isinstance(keywords, str) and keywords.startswith('[') and keywords.endswith(']'):
            import ast
            try:
                keywords = ast.literal_eval(keywords)
            except:
                keywords = [keywords]
        elif not isinstance(keywords, list):
            keywords = [keywords]
            
        all_keywords = "\n".join([k for k in keywords if k and isinstance(k, str)])
        if all_keywords:
            keyword_doc = Document(
                page_content=all_keywords,
                metadata={"manager": author}
            )
            keyword_documents.append(keyword_doc)

# 創建標題的 Chroma 向量資料庫
print("正在創建標題向量集合...")
title_vectorstore = Chroma.from_documents(
    documents=title_documents, 
    embedding=embeddings,
    persist_directory=db_path,
    collection_name="titles",  # 指定集合名稱
    ids=[doc.metadata["manager"] for doc in title_documents]
)
title_vectorstore.persist()
print(f"標題向量集合創建完成，包含 {len(title_documents)} 個文檔")

# 創建關鍵字的 Chroma 向量資料庫 (使用相同的 persist_directory)
print("正在創建關鍵字向量集合...")
keyword_vectorstore = Chroma.from_documents(
    documents=keyword_documents, 
    embedding=embeddings,
    persist_directory=db_path,
    collection_name="keywords",  # 指定不同的集合名稱
    ids=[doc.metadata["manager"] for doc in keyword_documents]
)
keyword_vectorstore.persist()
print(f"關鍵字向量集合創建完成，包含 {len(keyword_documents)} 個文檔")

# 驗證向量數據庫中的文檔數量
print("驗證向量數據庫...")
client = chromadb.PersistentClient(path=db_path)
title_collection = client.get_collection("titles")
keyword_collection = client.get_collection("keywords")

print(f"標題向量集合中的文檔數量: {title_collection.count()}")
print(f"關鍵字向量集合中的文檔數量: {keyword_collection.count()}")

print("向量資料庫已成功保存到 database 目錄")


正在清理現有的向量數據庫...
刪除現有的向量數據庫: database/vectorstore_bge_merged


PermissionError: [WinError 32] 程序無法存取檔案，因為檔案正由另一個程序使用。: 'database/vectorstore_bge_merged\\chroma.sqlite3'